# Stage 0 sweep on Colab (T4)

Runs a sweep grid (default **G-C**, the weight decay scouting grid: 84 runs) and writes a small results file you can hand over.

**Steps:** Runtime > Change runtime type > T4 GPU. Fill in `COMMIT` in the next cell. Then run the cells top to bottom. Results are saved to your Google Drive after every batch of runs, so if Colab disconnects, run the cells again and it resumes where it stopped.

**Sharing a grid between machines:** use the same `SPLIT` on every machine and a different `PART` on each (for example `"7,3"` with part 0 here and `--part 1` on your PC). Leave `SPLIT = "1"` to run everything here.

The code is pinned to one commit so every machine runs identical code. Training uses full float32 on purpose: switching to half precision would be faster but would change the numbers and break the comparison with the Stage 0 baseline.

In [ ]:
# ---- settings: edit these ----
REPO_URL  = "https://github.com/adi5022/Improving-Parameter-Efficiency-in-Transformer-Language-Models-Through-Superposition-Guided-Training.git"
COMMIT    = ""        # full commit hash with the G-C config (run `git rev-parse HEAD` on your PC after you commit and push)
GRID      = "G-C"     # G-C = scouting grid; G-A / G-B also work
SPLIT     = "1"       # relative speeds of all machines, e.g. "7,3"; "1" = this machine does everything
PART      = 0         # which share this notebook runs (0-based)
DRIVE_DIR = "/content/drive/MyDrive/superposition_results"
# If the repository is private, add a GitHub token in Colab Secrets (key icon, left side) named GH_TOKEN.
assert COMMIT, "Fill in COMMIT first"

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type"
print("torch", torch.__version__)

In [ ]:
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                      # no token: fine for a public repository
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", url, "/content/repo"], check=True)
%cd /content/repo
!git fetch --all -q
!git checkout -q {COMMIT}
!git rev-parse HEAD
!git status --short toy configs
!pip -q install numpy matplotlib

In [ ]:
# Keep results on Drive so a disconnect doesn't lose them (results/ is git-ignored, so the checkout stays clean).
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
!rm -rf results
os.symlink(DRIVE_DIR, "results")
!ls results

In [ ]:
# Quick speed check (about 30 seconds): trains 100 steps per width and estimates this machine's total time.
import time
from dataclasses import replace
from toy.batched import train_group
from toy.sweep import grid_configs, split_configs, make_groups

mine = split_configs(grid_configs(GRID), [float(w) for w in SPLIT.split(",")], PART)
groups = make_groups(mine, 24)
ms = {}
for g in groups:
    probe = [replace(c, steps=100, warmup_steps=20, test_batches=1, log_every=100) for c in g]
    train_group(probe[:2], device="cuda")                         # warm-up
    torch.cuda.synchronize(); t = time.time()
    train_group(probe, device="cuda"); torch.cuda.synchronize()
    ms[g[0].width] = (time.time() - t) / 100 * 1000
    print(f"width {g[0].width:>3}: {len(g)} runs together, {ms[g[0].width]:.1f} ms per step")
est = sum(ms[g[0].width] * g[0].steps / 1000 for g in groups) / 60
print(f"\n{len(mine)} runs here, estimated {est:.0f} min (rough; the 100-step test includes some setup cost)")

In [ ]:
# See what this notebook will run (trains nothing).
!python -m toy.sweep --grid {GRID} --split {SPLIT} --part {PART} --dry-run | tail -3

In [ ]:
# Run it. Safe to run again after a disconnect: finished runs are skipped.
!python -m toy.sweep --grid {GRID} --split {SPLIT} --part {PART}

## Results file

The next cell prints a table and makes **two files**:

- `<grid>_part<N>_small.zip`: just the run summaries and a report (a few KB). **This is the one to hand over.**
- `<grid>_part<N>_full.zip`: every run including the learned weights (tens of MB), only needed for the weight plots.

Save the small zip into your project (for example `results/colab/`) so it can be read.

In [ ]:
import os, json, shutil, collections, statistics, platform, datetime
out = f"{DRIVE_DIR}/stage0/{GRID}"
rows = [json.loads(l) for l in open(f"{out}/summary.jsonl") if l.strip()]
by = collections.defaultdict(list)
for r in rows:
    by[(r["width"], r["gamma"])].append(r["test_loss"])
gammas = sorted({k[1] for k in by})
print("mean test error over seeds (relative s.d. in brackets)")
print("width | " + " | ".join(f"g={g:+.2f}" for g in gammas))
for w in sorted({k[0] for k in by}):
    cells = []
    for g in gammas:
        v = by.get((w, g))
        cells.append("   -   " if not v else f"{statistics.mean(v):.2e} ({100*statistics.pstdev(v)/statistics.mean(v):.0f}%)")
    print(f"{w:>5} | " + " | ".join(cells))

report = {
    "grid": GRID, "split": SPLIT, "part": PART, "commit": COMMIT,
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "python": platform.python_version(),
    "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "runs_done": len(rows), "runs_not_ok": [r["run_id"] for r in rows if r["status"] != "ok"],
    "total_group_minutes": round(sum(r["train_seconds"] for r in rows) / 60, 1),
    "estimated_minutes_from_benchmark": round(est, 1) if "est" in globals() else None,
}
print("\n", json.dumps(report, indent=1))
tag = f"{GRID}_part{PART}"
json.dump(report, open(f"{out}/report.json", "w"), indent=2)
os.makedirs("/content/small", exist_ok=True)
shutil.copy(f"{out}/summary.jsonl", "/content/small/summary.jsonl")
shutil.copy(f"{out}/report.json", "/content/small/report.json")
shutil.make_archive(f"/content/{tag}_small", "zip", "/content/small")
shutil.make_archive(f"/content/{tag}_full", "zip", out)
from google.colab import files
files.download(f"/content/{tag}_small.zip")
files.download(f"/content/{tag}_full.zip")